# core

> the vault: one SQLite file holding everything you have read, and the retrieval over it

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A `Vault` is a [litesearch](https://github.com/Karthik777/litesearch) document tree — docs → nodes →
chunks, FTS5 and a usearch HNSW index — with an entity graph over it, in one SQLite file. Everything
you read goes in under a `kind`, and one query crosses all of them. Nothing here reimplements
litesearch: the vault holds an encoder and a store name, and hands both to it.

In [ ]:
#| export
import json, re, time, uuid, warnings
from functools import partial
import numpy as np
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from litesearch import (database, dir2files, doc_encoder, query_encoder, hash_embed, static_embedder,
        pdf_parse, build_graph, resolve_entities, FastEncode, DOC_EXTS,
        bge_model, embedding_gemma, modernbert, nomic_text_v15)

In [ ]:
#| export
KINDS = ('web', 'pdf', 'arxiv', 'youtube', 'file', 'code', 'data', 'note')
_window, DFLT_ENC = re.compile(r'^Pages \d+(?:–\d+)?:'), 'minishlab/potion-multilingual-128M'

def tidy_bc(bc:str) -> str:
    "Drop `build_tree`'s `Pages n–m:` window placeholders from a breadcrumb: real nodes, noise in a citation."
    return ' › '.join(dict.fromkeys(p for p in map(str.strip, (bc or '').split('›')) if p and not _window.match(p)))

def kinds(kind) -> L:
    "A kind filter as a list — `'note'`, `'note,web'` and `['note','web']` all work."
    return L(kind.split(',') if isinstance(kind, str) else kind).filter()

ENCODERS = {
    # alias          what litesearch loads                    what it reads better than the default
    'default':       DFLT_ENC,
    'multilingual':  DFLT_ENC,                              # 100+ languages, Devanagari included
    'retrieval':     'minishlab/potion-retrieval-32M',      # tuned for search, not for similarity
    'science':       'minishlab/potion-science-32M',        # papers: abstracts, methods, results
    'code':          'minishlab/potion-code-16M-v2',        # identifiers; what kosha embeds with
    'gemma':         embedding_gemma,                       # ONNX, ~300M: the most faithful, the slowest
    'bge-micro':     bge_model,                             # ONNX, quantized and tiny
    'modernbert':    modernbert,                            # ONNX, 8k context
    'nomic':         nomic_text_v15,                        # ONNX, asymmetric query/document prompts
}

def enc_spec(model=None) -> tuple:
    """`(what to load, how)` for an encoder: an `ENCODERS` alias, a model2vec id, a litesearch model
    dict, or an embedder you built yourself.

    The two `how`s are the two kinds of encoder litesearch has, and they are a real trade, not a
    detail: a static model is a lookup table — milliseconds per document, no GPU, no batch — while an
    ONNX transformer actually reads word order and costs perhaps a hundred times more per chunk.
    Ingest is where that bill lands, so the default stays static and the choice stays yours."""
    # `str` has an `.encode` too, and a litesearch model dict is a `dict`: both must fall through to
    # being *named*, not mistaken for a built embedder
    if model is not None and not isinstance(model, (str, bytes, dict)) and hasattr(model, 'encode'):
        return model, 'ready'
    spec = ENCODERS.get(model, model if model is not None else DFLT_ENC)
    return spec, ('onnx' if isinstance(spec, dict) else 'static')

def _is_enc(o) -> bool:
    "Is this already an `mk_encoder` result, rather than a name or one of litesearch's model dicts?"
    return isinstance(o, AttrDict) and 'doc' in o and 'query' in o

In [ ]:
#| export
def mk_encoder(model=None,          # an ENCODERS alias, a model2vec id, a litesearch model dict, or an embedder
               dims:int=256,        # dims for the hashing fallback only
               offline:bool=False,  # skip the download attempt entirely
               dtype=np.float16,    # stored width; litesearch's default everywhere
) -> AttrDict:
    """The best encoder available as `AttrDict(model, doc, query, dtype, dims, method, name, note)`.

    Degrades to litesearch's `hash_embed` rather than failing, and always says which answered: the
    gap between the two is the gap between a vault that answers questions and one that can only
    keyword-match. `model` is the embedder object itself, so kosha can share it (see `code`).

    Both encoders are cast to `dtype` because litesearch's own default is float16 and not every
    entry point takes a `dtype=`: `Database.context` does not thread one down to its section search,
    so a float32 store would be read back as float16 there — the vectors survive, the *distances*
    do not, and `context()` silently degrades to keyword ranking. Half precision costs nothing at
    these magnitudes; a mismatched width costs the whole semantic leg."""
    spec, how = enc_spec(model)
    nm = spec['model'] if isinstance(spec, dict) else (spec if isinstance(spec, str) else type(spec).__name__)
    if not offline:
        try:
            m = spec if how == 'ready' else FastEncode(spec, dtype=dtype) if how == 'onnx' else static_embedder(spec)
            v = m.encode(['probe'])
            cast = lambda f: lambda xs: np.asarray(f(xs), dtype=dtype)
            meth = 'onnx' if isinstance(m, FastEncode) else 'model2vec'
            return AttrDict(model=m, doc=cast(doc_encoder(m)), query=cast(query_encoder(m)), dtype=dtype,
                            dims=int(v.shape[-1]), method=meth, name=nm,
                            note=f'{nm} ({v.shape[-1]}d, {np.dtype(dtype)}, {meth})')
        except Exception as e:
            warnings.warn(f'could not load {nm} ({type(e).__name__}: {str(e)[:100]}); '
                          f'falling back to hash_embed — retrieval will be lexical, not semantic')
    f = partial(hash_embed, ndim=dims, dtype=dtype)
    return AttrDict(model=None, doc=f, query=f, dtype=dtype, dims=dims, method='hash', name='hash',
                    note=f'char-n-gram hashing ({dims}d) — lexical only; pass encoder= or restore '
                         f'network access for real semantics')

In [ ]:
#| export
class Vault:
    """Everything you have read, in one SQLite file, searchable as one corpus.

    Web pages, PDFs, papers, transcripts, local files, code and your own notes land in the same
    litesearch store under different `kind`s, which is the point: one query crosses all of them and
    `context()` hands back sections rather than fragments. Acquisition lives in
    `vishalakshi.acquire`, answering in `.ask`, code in `.code` — all optional, since the vault
    itself needs neither a network nor an LLM."""

    def __init__(self,
                 path:str=None,       # vault file; None -> ~/.vishalakshi/vault.db
                 encoder=None,        # an ENCODERS alias, a model id, an mk_encoder() result, or None
                 store:str='store',   # chunk store name
                 offline:bool=False,  # never attempt a model download
                 dims:int=256,        # dims for the hashing fallback
                 db=None):            # an open litesearch Database to share; shelves pass the vault's
        self.path = str(ifnone(path, Path.home()/'.vishalakshi'/'vault.db'))
        self.store = store
        # `_is_enc` rather than `isinstance(encoder, AttrDict)`: litesearch's own model dicts are
        # AttrDicts too, and mistaking one for a built encoder would leave the vault with no `doc`
        self.enc = encoder if _is_enc(encoder) else mk_encoder(encoder, dims=dims, offline=offline)
        self.dtype = self.enc.dtype
        # a shelf shares the connection rather than opening a second one, which is what makes shelves
        # work on ':memory:' at all — two `database(':memory:')` calls are two unrelated databases
        self.db = db if db is not None else database(self.path)
        self.g = self.db.get_tree(store, dtype=self.dtype, ndim=self.enc.dims)
        self._register()

    def qv(self, q:str) -> bytes:
        'Query-side embedding of `q`, as the bytes every litesearch search call wants.'
        return np.asarray(self.enc.query([q])[0], dtype=self.dtype).tobytes()

    def _where(self, kind) -> str:
        'A chunk-store `WHERE` for a kind filter — pushed into the search, not applied after it.'
        return None if not kinds(kind) else f'doc_id IN (SELECT id FROM {self.g.prefix}docs WHERE {_kw(kind)})'

    def __repr__(self):
        s = self.stats()
        return (f"Vault({self.path!r}: {s['docs']} docs, {s['chunks']} chunks, "
                f"{s['entities']} entities, encoder={self.enc.method})")

def _kw(kind) -> str: return 'kind IN (%s)' % ','.join(map(repr, kinds(kind)))

In [ ]:
#| export
@patch
def add(self:Vault,
        pages,                # markdown/text, or [(page_no, text)]
        title:str,            # document title
        source:str=None,      # url or path; defaults to the title. Identity is hashed over it
        kind:str='file',      # one of KINDS — the facet you filter and report on
        meta:dict=None,       # provenance: the query that found it, when, which tier fetched it
        force:bool=False,     # re-ingest a source already present
        **kw                  # forwarded to litesearch add_doc (chunker, summarize, with_heading)
) -> dict:
    """Ingest one document into the vault: tree, chunks, embeddings, ANN index.

    Identity is content-addressed over `source|title`, so re-adding the same page is a no-op rather
    than a duplicate — which is what makes it safe to re-run a search whose results overlap what
    you already have."""
    return self.db.add_doc(pages, title, source=source, kind=kind, store=self.store,
                           emb_fn=self.enc.doc, meta=meta, force=force, **kw)

@patch
def assets(self:Vault, name:str=None) -> Path:
    'Where extracted assets (PDF images) go: beside the vault file, never the working directory.'
    d = Path(self.path).parent/'assets'
    d.mkdir(parents=True, exist_ok=True)
    return d/name if name else d

@patch
def add_file(self:Vault, path:str, title:str=None, kind:str=None, **kw) -> dict:
    """Ingest one local file: PDFs page by page, everything else through litesearch's parsers.

    PDFs are parsed here rather than through `litesearch.add_file` for one reason: pdf-oxide writes
    extracted images relative to its `out_path`, which defaults to `./pdfs/`, so ingesting a paper
    would silently litter whatever directory you happened to be in. They go next to the vault."""
    p = Path(path)
    if p.suffix.lower() == '.pdf':
        return self.add(list(enumerate(pdf_parse(str(p), out_path=self.assets(p.stem)))),
                        title or p.stem.replace('_', ' ').replace('-', ' '),
                        source=str(p), kind=kind or 'pdf', **kw)
    r = self.db.add_file(p, title=title, store=self.store, emb_fn=self.enc.doc, **kw)
    if kind and r.get('doc_id') and not r.get('skipped'): self.g.docs.update(dict(id=r['doc_id'], kind=kind))
    return r

@patch
def add_dir(self:Vault, dir:str, types:str=DOC_EXTS, kind:str=None, **kw) -> L:
    'Ingest every document under a directory. `dir2files` skips dotfiles, tests, build and dist.'
    return dir2files(dir, types=types).map(self.add_file, kind=kind, **kw)

@patch
def note(self:Vault,
         text:str,            # what you want to remember
         title:str=None,      # defaults to the first line
         tags:list=None,      # free-form tags, kept in the doc's meta
) -> dict:
    """Write a note into the vault so it is searched alongside the corpus.

    Notes are ordinary documents with `kind='note'`, which is deliberate: the graph, the clusters
    and `context()` all see them for free, so what you concluded about a corpus comes back next to
    the evidence you concluded it from."""
    ttl = title or (text.strip().splitlines() or ['note'])[0].lstrip('# ')[:80]
    return self.add(text.strip(), ttl, source=f'note:{uuid.uuid4().hex[:12]}', kind='note',
                    meta=dict(tags=list(tags or [])))

In [ ]:
#| export
@patch
def find(self:Vault,
         q:str,              # query
         limit:int=10,       # hits to return
         kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
         **kw                # forwarded to litesearch doc_search
) -> list:
    'Chunk-level hybrid search (FTS5 + vectors, RRF-fused), each hit carrying its breadcrumb.'
    hits = self.db.doc_search(q, self.qv(q), limit=limit, store=self.store, dtype=self.dtype,
                              where=self._where(kind), **kw)
    for h in hits: h['breadcrumb'] = tidy_bc(h.get('breadcrumb'))
    return hits

@patch
def sections(self:Vault, q:str, limit:int=5, kind:str=None, per:int=3, **kw) -> list:
    'Ranked *sections* rather than chunks — the unit worth reading, each with a `read` handle.'
    secs = self.db.sections(q, self.qv(q), limit=limit, per=per, store=self.store, dtype=self.dtype,
                            where=self._where(kind), **kw)
    for s in secs: s['breadcrumb'] = tidy_bc(s.get('breadcrumb'))
    return secs

@patch
def context(self:Vault,
            q:str,              # the question
            sections:int=6,     # operative sections returned
            related:int=8,      # related sections reached by graph + vector
            kind:str=None,      # restrict to one or more KINDS ('note' or 'note,web')
            max_read:int=6000,  # chars of assembled text per section
            code:int=None,      # code sections to append; None -> 4 if kosha has indexed the repo
            shelves:int=2,      # sections to append from each *other* shelf; 0 -> none
            dir:str=None,       # repo for the code legs; None -> the cwd repo
            **kw                # forwarded to litesearch context
) -> AttrDict:
    """The retrieval an LLM should be handed: whole sections plus what they connect to.

    Operative sections carry `text, breadcrumb, pages, filename` and their tree neighbourhood;
    `related` holds sections reached by the entity graph (`via='graph'`) and by embedding
    similarity (`via='vector'`). `kind` filters *after* retrieval here, because litesearch's
    `context` does not thread a `where` down to both legs — the over-fetch covers the common case,
    but a filter matching very little of a large vault can still come back short.

    A question about your own system is usually answered by the source rather than by prose about it,
    so `code=None` appends federated code sections *when kosha has already indexed the repo* —
    evidence you want them, and a file check rather than a model load to find out. They go after the
    prose sections, so the `[n]` numbering an answer cites is unaffected by whether the leg ran."""
    keep = None if not kinds(kind) else {r['id'] for r in self.g.docs(where=_kw(kind), select='id')}
    ctx = self.db.context(q, self.qv(q), store=self.store, related=related, max_read=max_read,
                          sections=sections*3 if keep else sections, **kw)
    if keep is not None:
        ctx.results = ctx.results.filter(lambda r: r.doc_id in keep)[:sections]
        ctx.related = ctx.related.filter(lambda r: r.doc_id in keep)[:related]
    for r in (*ctx.results, *ctx.related): r.breadcrumb = tidy_bc(r.breadcrumb)
    ctx.encoder, ctx.code, ctx.shelves = self.enc.note, 0, 0
    if shelves:
        found = self.elsewhere(q, limit=shelves)
        ctx.results, ctx.shelves = ctx.results + found, len(found)
    if code or code is None:
        # absolute, not relative: this same line has to run in the notebook, where there is no parent
        # package to resolve `.code` against. The module is optional — kosha and ripgrep live there.
        from vishalakshi.code import code_sections, kosha_indexed
        if code or kosha_indexed(dir):
            hits = code_sections(self, q, n=code or 4, dir=dir)
            ctx.results, ctx.code = ctx.results + hits, len(hits)
    return ctx

@patch
def related(self:Vault, node_id:str, limit:int=8) -> L:
    """Sections nearest an existing one — "what else in the vault reads like this".

    Reuses the vectors usearch already holds, so nothing is re-embedded."""
    out = {}
    for r in self.g.store(where=f'node_id={node_id!r}', select='rowid as rowid'):
        for n in self.g.store.ann_neighbors(r['rowid'], limit=limit*3, dtype=self.dtype,
                                            columns=['content', 'node_id', 'doc_id']):
            nid = n.get('node_id')
            if nid and nid != node_id and nid not in out:
                out[nid] = dict(node_id=nid, doc_id=n.get('doc_id'), dist=n.get('_dist'),
                                breadcrumb=tidy_bc(self.db.breadcrumb(nid, self.store)),
                                snippet=(n.get('content') or '')[:300])
            if len(out) >= limit: return L(out.values())
    return L(out.values())

@patch
def read(self:Vault, node_id:str, max_chars:int=6000, store:str=None) -> dict:
    """Assemble a whole section back out of its chunks.

    `store` opens a section on another shelf, which is what a citation from `elsewhere` needs."""
    return self.db.read(node_id, store=store or self.store, max_chars=max_chars)

@patch
def toc(self:Vault, **kw) -> list:
    'The table of contents across every document in the vault.'
    return self.db.toc(store=self.store, **kw)

A `node_id` names a section and a `doc_id` names a document, so retrieval hands back the first and
acquisition the second. These three close the gap: one document row, one whole document, and a place
to record what you later concluded about it.

In [ ]:
#| export
@patch
def doc(self:Vault, ref:str) -> dict:
    """One document row, by `doc_id`, exact `source`, or a title substring; `meta` already decoded.

    Three ways to name one document because three different things hand you a reference: retrieval
    returns `doc_id`s, acquisition returns the url or path it filed, and you remember the title.
    Exact matches are tried first, so a title that happens to contain another's is still reachable."""
    q = str(ref or '').replace("'", "''")
    for w in (f"id='{q}'", f"source='{q}'", f"title LIKE '%{q}%'"):
        if (r := first(self.g.docs(where=w, order_by='added_at desc'))):
            return dict(r, meta=json.loads(r['meta'] or '{}'))
    return None

@patch
def document(self:Vault,
             ref:str,               # doc_id, source (url or path), or a title substring
             max_chars:int=40000,   # cap on the text returned
             headings:bool=True,    # put the node titles back as markdown headings
             disk:bool=True,        # fall back to a path on disk the vault has never seen
) -> AttrDict:
    """One whole document, reassembled in document order — the unit a model reads to extract from.

    `read` returns a section; this returns all of them. Headings go back in because a chunk is
    stored bare and a heading is often the only thing that says what a number *means*: an invoice
    reassembled without its `Total` line is a column of unlabelled figures — but only the headings the
    document actually wrote, never `build_tree`'s window placeholders. `disk=True` reads a markdown or
    source file the vault does not hold, so a file can be handed to a model without ingesting it
    first — `origin` says which of the two answered."""
    d = self.doc(ref)
    if d is None:
        p = Path(ref or '')
        if not (disk and p.is_file()): raise ValueError(f'no document in the vault matching {ref!r}')
        txt = p.read_text(errors='replace')
        return AttrDict(doc_id=None, title=p.name, source=str(p), kind='file', meta={}, pages=None,
                        origin='disk', nodes=0, chars=len(txt), truncated=len(txt) > max_chars,
                        text=txt[:max_chars])
    did = d['id'].replace("'", "''")
    chunks = {}
    for c in self.g.store(where=f"doc_id='{did}'", select='content, node_id, page, rowid as rowid'):
        chunks.setdefault(c['node_id'], []).append(c)
    nodes, parts = sorted(self.g.nodes(where=f"doc_id='{did}'"), key=lambda r: r['seq']), []
    for nd in nodes:
        # a `Pages n–m:` title is `build_tree`'s window placeholder, not a heading the document
        # wrote — noise here for the same reason `tidy_bc` drops it from a citation
        if headings and nd['level'] and (t := (nd['title'] or '').strip()) and not _window.match(t):
            parts.append('#'*min(nd['level'], 6) + ' ' + t)
        parts += [c['content'] for c in sorted(chunks.get(nd['id'], []),
                                               key=lambda c: (c['page'] or 0, c['rowid']))]
    txt = '\n\n'.join(p for p in parts if (p or '').strip())
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], kind=d['kind'], meta=d['meta'],
                    pages=d['pages'], origin='vault', nodes=len(nodes), chars=len(txt),
                    truncated=len(txt) > max_chars, text=txt[:max_chars])

@patch
def set_meta(self:Vault, doc_id:str, **kv) -> dict:
    """Merge key/values into one document's `meta`, and return the merged dict.

    What a document *is* lands here rather than in `kind`: `kind` is the fixed facet retrieval
    filters on (`KINDS`, chosen by whatever acquired the document), while a document type is a
    judgement — one a later, better model may revise."""
    did = (doc_id or '').replace("'", "''")
    r = first(self.g.docs(where=f"id='{did}'"))
    if not r: raise ValueError(f'no document {doc_id!r} in the vault')
    m = {**json.loads(r['meta'] or '{}'), **kv}
    self.g.docs.update(dict(id=doc_id, meta=json.dumps(m, default=str)))
    return m

One vault file, several vector spaces. A corpus of papers and a corpus of Sanskrit want different
embedders, and an encoder cannot be mixed *within* an index — so each gets a shelf, and results are
combined by rank rather than by distance.

In [ ]:
#| export
@patch
def _stores(self:Vault):
    'The registry of stores in this vault file and which encoder wrote each; created on first use.'
    t = self.db.t.vault_stores
    t.create(store=str, encoder=str, dims=int, method=str, added_at=float, pk='store', if_not_exists=True)
    return t

@patch
def _register(self:Vault):
    """Record which encoder wrote this store, and say so loudly when it is reopened with another.

    This is the one mistake here that is silent *and* total. litesearch warns when the stored vector
    *width* disagrees with the one being searched; when the width matches and the model does not —
    two 256-dim static models, or a real encoder replaced by the hashing fallback after a failed
    download — nothing raises, nothing warns, and every distance is computed over vectors from two
    different spaces. Recording the name at open is enough to catch it, so it is worth a table."""
    try:
        t, now = self._stores(), time.time()
        r = first(t(where=f'store={self.store!r}'))
        if r and (r['encoder'], r['dims']) != (self.enc.name, self.enc.dims):
            warnings.warn(
                f"store {self.store!r} was written by {r['encoder']} ({r['dims']}d) but this Vault is "
                f"using {self.enc.name} ({self.enc.dims}d). Distances across the two are meaningless. "
                f"Re-ingest, or keep them apart with shelf('{self.store}-{self.enc.method}').")
        elif not r:
            t.insert(dict(store=self.store, encoder=self.enc.name, dims=self.enc.dims,
                          method=self.enc.method, added_at=now), replace=True)
    except Exception: pass   # a read-only vault must still open; the registry is a convenience

@patch
def shelf(self:Vault, name:str, encoder:str=None, **kw) -> Vault:
    """A sibling vault in the same file: its own store, its own encoder, its own ANN index.

    This is how one file holds corpora that want different embedders — papers under a science model,
    Devanagari under a multilingual one, a contract archive under something that reads word order —
    and it is a *partition* rather than a mixture for a hard reason: one ANN index holds one vector
    space. Two models' vectors in one index get compared as bytes and the distances mean nothing.
    Partitioned, each shelf ranks correctly on its own, and `federate(shelves=…)` combines them by
    *rank*, which is the only thing that survives a change of encoder.

    With no `encoder`, the shelf is reopened with the one that wrote it."""
    was = first(self._stores()(where=f'store={name!r}')) or {}
    enc = encoder or was.get('encoder') or SHELVES.get(name)
    if enc == 'hash': enc, kw = None, dict(kw, offline=True)   # nothing to load; do not try
    return Vault(self.path, encoder=enc, store=name, db=self.db, **kw)

@patch
def shelves(self:Vault) -> L:
    'Every store in this vault file, with the encoder that wrote it and how many documents it holds.'
    def n(s):
        p = '' if s == 'store' else f'{s}_'
        try: return self.db.t[f'{p}docs'].count
        except Exception: return 0
    return L(self._stores()(order_by='added_at')).map(lambda r: dict(r, docs=n(r['store'])))

SHELVES = {
    # shelf       encoder      what belongs on it
    'store':      'default',   # the main shelf: notes, pages, anything unrouted
    'papers':     'science',   # arXiv and papers — potion-science reads abstracts and methods
    'sanskrit':   'gemma',     # Devanagari and transliterated prose: veda, commentary, translation
    'code':       'code',      # source filed as prose; kosha is the real code index, reached by federate
    'data':       'retrieval', # API harvests and record dumps: short, keyword-shaped text
}

# Where acquisition puts things. Only the certain cases: a PDF is as likely an invoice as a paper,
# and `categorize` is what knows the difference — after the fact, which is too late for a route.
KIND_SHELF = {'arxiv': 'papers', 'code': 'code', 'data': 'data'}

@patch
def route(self:Vault, kind:str) -> Vault:
    "The shelf a `kind` belongs on: `self`, unless `KIND_SHELF` sends it elsewhere."
    nm = KIND_SHELF.get(kind)
    return self.shelf(nm) if nm and nm != self.store else self

@patch
def elsewhere(self:Vault,
              q:str,              # the question
              limit:int=2,        # sections taken from each other shelf
              shelves=True,       # True -> every other shelf; a list picks some
              max_read:int=2000,  # chars kept per section
) -> L:
    """Sections from the vault's *other* shelves, shaped like this one's.

    Each shelf has its own encoder, so its scores are not comparable with this shelf's — which is why
    these arrive as extra sections for a reader to weigh rather than merged into one ranking. `store`
    rides along on every row, because that is what `read(node_id, store=…)` needs to open one."""
    out, want = L(), (None if shelves is True else set(L(shelves)))
    for s in self.shelves():
        nm = s['store']
        if nm == self.store or (want is not None and nm not in want): continue
        for r in self.shelf(nm).sections(q, limit=limit):
            out.append(AttrDict(node_id=r['node_id'], doc_id=r['node_id'].split('#')[0], store=nm,
                                title=r['title'], breadcrumb=f"{nm} › {r['breadcrumb']}", filename=None,
                                pages=r['pages'], via=f'shelf:{nm}',
                                text=' '.join(r['snippets'])[:max_read]))
    return out

In [ ]:
#| export
@patch
def connect(self:Vault, resolve:bool=True, **kw) -> dict:
    '(Re)build the entity graph over everything in the vault.'
    chunks = list(self.g.store())
    if not chunks: return dict(entities=0, mentions=0, edges=0, windows=0)
    self.db.get_graph(self.store, ndim=self.enc.dims, dtype=self.dtype)
    res = build_graph(self.db, chunks, store=self.store, emb_fn=self.enc.doc, **kw)
    if resolve: res = dict(res, resolved=resolve_entities(self.db, store=self.store, dtype=self.dtype))
    return res

@patch
def map(self:Vault, min_count:int=2, **kw) -> AttrDict:
    'Cluster the corpus into labelled topics — the shape of what you have collected.'
    return self.g.store.clusters(min_count=min_count, dtype=self.dtype, columns=['content', 'doc_id'], **kw)

@patch
def sources(self:Vault, kind:str=None) -> L:
    'Every document in the vault with its provenance, newest first.'
    rows = self.g.docs(where=_kw(kind) if kinds(kind) else None, order_by='added_at desc')
    return L(rows).map(lambda d: dict(d, meta=json.loads(d['meta'] or '{}')))

@patch
def forget(self:Vault, doc_id:str):
    'Remove a document, its sections and its chunks, and rebuild the ANN index.'
    self.db.delete_doc(doc_id, store=self.store)

@patch
def stats(self:Vault) -> dict:
    'Row counts across the vault, by kind.'
    p, t = self.g.prefix, self.db.t
    return dict(docs=self.g.docs.count, nodes=self.g.nodes.count, chunks=self.g.store.count,
                entities=t[f'{p}entities'].count if f'{p}entities' in t else 0,
                by_kind={r['kind']: r['n'] for r in
                         self.db.q(f'select kind, count(*) as n from {p}docs group by kind order by n desc')},
                encoder=self.enc.method, path=self.path)

## Try it

`offline=True` skips the model download and uses litesearch's `hash_embed`, which is what you want
in CI and for a quick look: retrieval is lexical, and `stats()['encoder']` says so.

In [ ]:
v = Vault(':memory:')
v.note('Late chunking beats naive chunking because context survives the split.', tags=['retrieval'])
v.add('# Attention\n\nScaled dot-product attention weights values by query-key similarity.\n\n'
      '## Multi-head\n\nHeads attend to different subspaces in parallel.', 'Attention', kind='note')
v.stats()

{'docs': 2,
 'nodes': 5,
 'chunks': 3,
 'entities': 0,
 'by_kind': {'note': 2},
 'encoder': 'model2vec',
 'path': ':memory:'}

In [ ]:
test_eq(v.stats()['docs'], 2)
test_eq(v.stats()['encoder'], 'model2vec')
assert v.find('chunking')[0]['content']
test_eq([d['kind'] for d in v.sources()], ['note', 'note'])
test_eq(len(v.sources(kind='web')), 0)
test_eq(len(v.find('chunking', kind='web')), 0)

In [ ]:
d = v.doc('Attention')                          # by title substring
test_eq(v.doc(d['id'])['title'], 'Attention')   # by doc_id
test_eq(v.doc(d['source'])['title'], 'Attention')  # by source
test_eq(v.doc('nothing in here'), None)

whole = v.document('Attention')
test_eq((whole.origin, whole.nodes), ('vault', 3))   # the root, the heading, the subheading
# every section, in document order, with the headings that say what each one is
assert whole.text.index('Scaled dot-product') < whole.text.index('## Multi-head') < whole.text.index('subspaces')
test_eq(v.document('Attention', headings=False).text.find('## Multi-head'), -1)
test_eq(v.document('Attention', max_chars=20).text, whole.text[:20])
test_eq(v.document('Attention', max_chars=20).truncated, True)
# a `Pages n–m:` node title is build_tree's placeholder, not a heading the document wrote
v.add('Prose with no headings at all, long enough to chunk and to be stored in the vault.', 'plain')
assert '# Pages' not in v.document('plain').text and 'Prose with no' in v.document('plain').text

test_eq(v.set_meta(d['id'], doctype='paper')['doctype'], 'paper')
test_eq(v.doc(d['id'])['meta']['doctype'], 'paper')                 # survives the round trip
test_eq(v.set_meta(d['id'], reviewed=True)['doctype'], 'paper')     # merged, not replaced
test_fail(lambda: v.document('no such document'), contains='no document in the vault')

In [ ]:
#| hide
# resolving an encoder name is pure, so this costs no download: the alias, the kind, and the id
test_eq(enc_spec('science'), ('minishlab/potion-science-32M', 'static'))
test_eq(enc_spec('gemma')[1], 'onnx')
test_eq(enc_spec('gemma')[0]['model'], 'onnx-community/embeddinggemma-300m-ONNX')
test_eq(enc_spec(None)[0], DFLT_ENC)
test_eq(enc_spec('some-org/some-model'), ('some-org/some-model', 'static'))   # an id passes through

class _E:            # an embedder you built yourself: litesearch's doc_encoder takes anything with .encode
    def encode(self, xs): return np.zeros((len(xs), 8), dtype=np.float16)
test_eq(enc_spec(_E())[1], 'ready')
test_eq((mk_encoder(_E()).dims, mk_encoder(_E()).method), (8, 'model2vec'))
# litesearch's model dicts are AttrDicts too, so `Vault(encoder=embedding_gemma)` must not be read as
# an already-built encoder — that would leave the vault with no `doc` function at all
assert not _is_enc(embedding_gemma) and _is_enc(mk_encoder(_E()))
test_eq(mk_encoder('no/such-model-at-all', offline=True).method, 'hash')

In [ ]:
#| hide
# a shelf is a partition of one file: its own store, its own encoder, one shared connection
sh = v.shelf('papers', offline=True)
sh.add('Contextual chunk embeddings keep the document around the chunk.', 'a paper')
test_eq((sh.store, sh.db is v.db), ('papers', True))    # shared, or ':memory:' would be a second db
test_eq([s['store'] for s in v.shelves()], ['store', 'papers'])
test_eq(v.doc('a paper'), None)                         # a partition, not a second index over the same docs
test_eq(sh.doc('a paper')['title'], 'a paper')
test_eq(v.shelf('papers').enc.method, 'hash')           # reopened with the encoder that wrote it
test_eq(v.shelf('sanskrit', offline=True).store, 'sanskrit')
test_eq(SHELVES['papers'], 'science')                   # a new shelf takes its encoder from the registry
test_eq((v.route('arxiv').store, v.route('web').store), ('papers', 'store'))

# reopening a store with a *different* encoder is silent and total, so it has to be said out loud
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    v.shelf('papers', encoder=_E())
    assert any('meaningless' in str(x.message) for x in w), [str(x.message) for x in w]

# ...and what makes a library of shelves usable is reading across it
e = v.elsewhere('chunk embeddings')
test_eq(e.attrgot('store'), ['papers'])
assert e[0].breadcrumb.startswith('papers › ') and e[0].text
assert v.read(e[0].node_id, store='papers')['text']   # a citation into a shelf has to open
test_eq(v.read(e[0].node_id), {})                     # and the wrong store finds nothing
test_eq(v.elsewhere('chunk embeddings', shelves=['nowhere']), [])
test_eq(sh.elsewhere('late chunking', limit=1).attrgot('store'), ['store'])   # reads both ways
test_eq(len(sh.elsewhere('late chunking', limit=2)), 2)                      # limit is per shelf

c = v.context('chunk embeddings', sections=2, related=0, code=0)
test_eq(c.shelves, 1)
assert any(r.get('store') == 'papers' for r in c.results), c.results
test_eq(v.context('chunk embeddings', related=0, code=0, shelves=0).shelves, 0)

In [ ]:
test_eq(tidy_bc('Attention › Pages 1–1: Scaled dot-product › Multi-head'), 'Attention › Multi-head')
test_eq(tidy_bc(None), '')
# a window with a blank first line leaves the placeholder bare, and it is still a placeholder
test_eq(tidy_bc('Doc › Pages 1–1: › Body'), 'Doc › Body')

In [ ]:
r = v.connect()
assert r['entities'] > 0 and r['resolved']['resolvable'] == r['entities']
test_eq(v.stats()['entities'], r['entities'])

In [ ]:
#| hide
# the store, the query vector and litesearch's own default must all agree on width, or `context`
# reads f32 bytes as f16 and ranks by keyword alone
test_eq(v.enc.dtype, np.float16)
test_eq(len(v.qv('chunking')), v.enc.dims*2)
with warnings.catch_warnings():
    warnings.simplefilter('error')          # litesearch warns on a dtype mismatch; it must not fire
    assert v.context('why does late chunking help', sections=2, related=2,
                     code=0, shelves=0).results   # this cell is about *this* store's width

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()